# Deep Q-Learning for Radiology Worklist Triage

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 9: Deep Reinforcement Learning**

Reading order is a sequential decision problem: the value of reading a study now
depends on what else is waiting and how long each has waited. A DQN learns a
policy over a simulated reading room and is compared against first-in-first-out
and against a clinical heuristic.

The reward penalises time-to-read weighted by true urgency, so the agent is
optimised for *harm avoided*, not for classification accuracy. The trained policy's
final linear layer is exported for serving — a dot product on Render's 0.1 CPU
rather than a network forward pass.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. The environment

In [ ]:
from collections import deque
import random

URGENCY = {"Pneumothorax":1.00,"Edema":0.85,"Consolidation":0.70,"Pneumonia":0.70,
           "Mass":0.65,"Effusion":0.55,"Cardiomegaly":0.45,"Infiltration":0.45,
           "Nodule":0.40,"Atelectasis":0.35,"Pleural_Thickening":0.25,
           "Fibrosis":0.20,"Emphysema":0.20,"Hernia":0.15}
U = np.array([URGENCY[p] for p in PATHOLOGIES])

class ReadingRoom:
    """Studies arrive stochastically; the agent picks which to read next.

    The reward is a PURE COST model: every step, each study still waiting costs
    its clinical urgency. The agent reads exactly one study per step, so the
    only way to reduce cost is to remove the most expensive study from the
    queue first.

    An earlier version of this environment also paid a bonus for reading. That
    bonus dominated the waiting cost, so random, FIFO and urgency-greedy
    policies all scored within noise of each other — the environment could not
    distinguish a good policy from a coin flip, which makes it useless as a
    benchmark. Removing the bonus makes ordering the only thing that matters.

    The agent is never rewarded for being right about a diagnosis; that is the
    classifier's job. It is rewarded only for reading in a good order.
    """

    def __init__(self, queue_size=10, horizon=200, arrival_rate=1.6, seed=SEED):
        self.queue_size, self.horizon = queue_size, horizon
        self.arrival_rate = arrival_rate   # >1 keeps the queue under pressure
        self.rng = np.random.default_rng(seed)

    def _new_study(self):
        probs = self.rng.beta(0.5, 8.0, 14)           # most studies unremarkable
        if self.rng.random() < 0.25:                   # 25% carry a real finding
            probs[self.rng.integers(0, 14)] = self.rng.uniform(0.55, 0.99)
        return {"probs": probs, "wait": 0.0,
                "urgency": float(np.dot(probs, U) / U.max())}

    def reset(self):
        self.t = 0
        self.queue = [self._new_study() for _ in range(self.queue_size)]
        return self._state()

    def _features(self, s):
        crit = max(s["probs"][PATHOLOGIES.index(p)]
                   for p in ["Pneumothorax","Edema","Consolidation","Pneumonia","Mass"])
        return np.array([crit, s["urgency"], min(s["wait"]/120.0, 1.0), 0.0, 0.0, 0.0,
                         min(len(self.queue)/50.0, 1.0)])

    def _state(self):
        f = [self._features(s) for s in self.queue[:self.queue_size]]
        while len(f) < self.queue_size: f.append(np.zeros(7))
        return np.stack(f)

    def step(self, action):
        # Read one study — it stops accruing cost from this step onward.
        if self.queue and action < len(self.queue):
            self.queue.pop(action)

        # Everything still waiting costs its urgency, every step.
        reward = -float(sum(s["urgency"] for s in self.queue))
        for s in self.queue:
            s["wait"] += 1.0

        # Poisson arrivals keep the queue full so there is always a choice.
        for _ in range(self.rng.poisson(self.arrival_rate)):
            if len(self.queue) < self.queue_size:
                self.queue.append(self._new_study())

        self.t += 1
        return self._state(), reward, self.t >= self.horizon

env = ReadingRoom(); s = env.reset()
print("state shape:", s.shape, "(queue_size x 7 features)")
print("reward is pure cost: sum of urgency over everything still waiting.")

## 2. The agent

Double DQN with a target network and experience replay — the three fixes that make Q-learning with function approximation stable.

In [ ]:
class QNet(nn.Module):
    """Scores each queued study. The final layer is what gets exported."""
    def __init__(self, n_features=7, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(),
                                   nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Linear(hidden, 1)
    def forward(self, x): return self.head(self.trunk(x)).squeeze(-1)

class DQNAgent:
    def __init__(self, lr=1e-3, gamma=0.95, buffer=20000, batch=64):
        self.q, self.target = QNet().to(DEVICE), QNet().to(DEVICE)
        self.target.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buf, self.gamma, self.batch = deque(maxlen=buffer), gamma, batch

    def act(self, state, eps):
        if random.random() < eps: return random.randrange(len(state))
        with torch.no_grad():
            return int(self.q(torch.as_tensor(state, dtype=torch.float32,
                                              device=DEVICE)).argmax())

    def learn(self):
        if len(self.buf) < self.batch: return None
        S, A, R, S2, D = zip(*random.sample(self.buf, self.batch))
        S  = torch.as_tensor(np.stack(S),  dtype=torch.float32, device=DEVICE)
        S2 = torch.as_tensor(np.stack(S2), dtype=torch.float32, device=DEVICE)
        R  = torch.as_tensor(R, dtype=torch.float32, device=DEVICE)
        D  = torch.as_tensor(D, dtype=torch.float32, device=DEVICE)
        A  = torch.as_tensor(A, dtype=torch.long, device=DEVICE)

        q = self.q(S).gather(1, A.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            # Double DQN: the ONLINE net picks the action, the TARGET net
            # values it. Using one net for both systematically over-estimates Q.
            best = self.q(S2).argmax(1, keepdim=True)
            q_next = self.target(S2).gather(1, best).squeeze(1)
            target = R + self.gamma * q_next * (1 - D)
        loss = F.smooth_l1_loss(q, target)   # Huber: robust to reward outliers
        self.opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()
        return float(loss)

print("Agent defined.")

## 3. Baselines and export

An RL agent that cannot beat a sensible heuristic has not earned its place in the system.

In [ ]:
def run_policy(env, choose, episodes=30):
    totals = []
    for _ in range(episodes):
        s, done, total = env.reset(), False, 0.0
        while not done:
            s, r, done = env.step(choose(s, env)); total += r
        totals.append(total)
    return float(np.mean(totals)), float(np.std(totals))

fifo      = lambda s, env: 0            # longest-waiting first
urgency_h = lambda s, env: int(np.argmax(s[:, 0] * 0.62 + s[:, 1] * 0.18))
oracle    = lambda s, env: int(np.argmax([x["urgency"] for x in env.queue]))
random_p  = lambda s, env: random.randrange(max(1, len(env.queue)))

print(f"{'policy':12s} {'mean return':>12s}  {'std':>6s}")
for name, pol in [("random", random_p), ("FIFO", fifo),
                  ("heuristic", urgency_h), ("oracle-greedy", oracle)]:
    m, sd = run_policy(ReadingRoom(seed=7), pol)
    print(f"{name:12s} {m:12.2f}  {sd:6.2f}")
print("\nThese must be clearly SEPARATED. If random and oracle score the same,")
print("the environment cannot distinguish policies and any DQN result on it")
print("would be meaningless. Oracle-greedy is the practical upper bound: it")
print("reads the highest-urgency study using ground truth the agent cannot see.")
print("\nTrain the DQN, then compare. Report the DQN ONLY if it beats the")
print("heuristic — otherwise the honest finding is that a simple clinical prior")
print("is sufficient, and that is a legitimate result worth reporting.")

def export_policy(agent, path="dqn_policy.json", episodes=0):
    """Export the final linear layer for cheap serving on Render."""
    import json
    W = agent.q.head.weight.detach().cpu().numpy().ravel()
    # The trunk is non-linear, so this is a linearisation, not an exact copy.
    # Validate rank correlation against the full network before deploying.
    probe = torch.eye(7, device=DEVICE)
    approx = agent.q(probe).detach().cpu().numpy()
    json.dump({"weights": approx.tolist(), "bias": 0.0, "episodes": episodes},
              open(path, "w"), indent=2)
    print(f"Wrote {path} -> copy to apps/api/artifacts/")

print("\nexport_policy ready.")

---

### References for this notebook

- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*.
- van Hasselt, H., Guez, A. & Silver, D. (2016). Deep RL with double Q-learning. *AAAI*.
- Vinyals, O. et al. (2019). Grandmaster level in StarCraft II. *Nature*.
- Badia, A. P. et al. (2020). Agent57: outperforming the Atari human benchmark. *ICML*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
